# ISA-LLM

## Imports

In [1]:
from dotenv import load_dotenv
import os
from openai import OpenAI
import json
import pandas as pd
from matplotlib import pyplot as plt


## Env

In [2]:
load_dotenv(override=True)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [ ]:
MODELS = [
    "gpt-5.4-nano-2026-03-17",
    "gpt-5.4-mini-2026-03-17",
    "gpt-5.4",
]

TEMPERATURE = 0

RUNS = 28


## Functions

In [ ]:
# def read_qas(file_path: str) -> list:
#     """
#     Read and preprocess the data from the given CSV file.

#     Args:
#         file_path (str): The path to the CSV file.
#     """

#     # read csv
#     df = pd.read_csv(file_path, sep=";")

#     # do some clean up
#     df = df[1:]
#     df.columns = ["SET", "CODE_d", "CODE_i", "CONTEXT QUESTION_d", "CONTEXT QUESTION_i", "CRITICAL_REPLY"]

#     # reshape dataframe
#     df_direct = df[["CODE_d", "CONTEXT QUESTION_d", "CRITICAL_REPLY"]]
#     df_direct.columns=["id", "Person_A", "Person_B"]
#     df_indirect = df[["CODE_i", "CONTEXT QUESTION_i", "CRITICAL_REPLY"]]
#     df_indirect.columns=["id", "Person_A", "Person_B"]
#     df_all = pd.concat([df_direct, df_indirect], axis=0)

#     # concatenate Person A and Person B into one column called "text" and drop the original columns
#     df_all["text"] = "Person A: " + df_all["Person_A"] + "; Person B: " + df_all["Person_B"]
#     df_all.drop(columns=["Person_A", "Person_B"], inplace=True)

#     # convert into json and save to file
#     output = [
#     {
#         "id": str(row["id"]),
#         "text": row["text"]
#     }
#     for _, row in df_all.iterrows()
#     ]

#     return output

def read_qas(file_path: str, retain=None) -> list:
    """
    Read the data from the given CSV file and transform it into a json structure that can be used as batch input to the LLMs.

    Args:
        file_path (str): The path to the CSV file.
        retain (int): The number of rows to include in the output for direct and indirect qaüpairs respectively.
    """
    # read the csv file
    df = pd.read_csv(file_path, sep=";")

    # clip if necessary
    if retain is not None:
        if retain > len(df):
            raise ValueError(f"Retain value {retain} is greater than the number of qas in the dataframe {len(df)}.")
        df = df.head(retain)


    # extract only relevant information
    df_new = df[["CODE", "CONTEXT QUESTION", "CRITICAL UTTERANCE"]]
    df_new["text"]  = "Person A: " + df_new["CONTEXT QUESTION"] + "; Person B: " + df_new["CRITICAL UTTERANCE"]
    df_new.rename(columns={"CODE": "id"}, inplace=True)

    # convert into json
    output = [
    {
        "id": str(row["id"]),
        "text": row["text"]
    }
    for _, row in df_new.iterrows()
    ]

    return output


def score_qas(model: str, qas: list) -> dict:
    """
    Score the QAs using the specified model.

    Args:
        model (str): The model to use for scoring.
        qas (list): The list of QAs to score.

    Returns:
        dict: The scoring results.
    """

    SYSTEM_PROMPT2 = '''
    Person A and Person B are taliking to each other. Person A asks a question and Person B answers. Given a Person A's question, estimate how strongly Person B's answer should be interpreted as answering “yes” versus “no.”

    Output a single integer score from 1 to 7, where:

    1 = definitely no
    2 = very likely no
    3 = somewhat likely no
    4 = completely uncertain, ambiguous
    5 = somewhat likely yes
    6 = very likely yes
    7 = definitely yes    

    Also report the rationale for your decision in a few sentences, explaining why you gave the score you did. If the answer is ambiguous or irrelevant, explain why.
    '''

    client = OpenAI()

    response = client.responses.create(
        model=model,
        temperature=TEMPERATURE,
        input=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT2,
            },
            {
                "role": "user",
                "content": json.dumps(qas),
            },
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": "batch_classification_scores",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {
                        "results": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "id": {
                                        "type": "string"
                                    },
                                    "score": {
                                        "type": "integer",
                                        "minimum": 1,
                                        "maximum": 7
                                    },
                                    "rationale": {
                                        "type": "string"
                                    }
                                },
                                "required": ["id", "score", "rationale"],
                                "additionalProperties": False
                            }
                        }
                    },
                    "required": ["results"],
                    "additionalProperties": False
                },
            }
        },
    )

    result = json.loads(response.output_text)

    return result


def save_result(model: str, run: int, result):
    '''Save the scoring results to a CSV file.'''
    
    # Convert results to a dataframe
    results_df = pd.DataFrame(result["results"])

    # Save to CSV
    results_df.to_csv(f"results/{model}_score_run{run+1}.csv", index=False)


In [8]:
qas = read_qas("data/iCO-Eval2_summarizedRatings.csv", retain=5)

for model in MODELS:
    for run in range(RUNS):
        result = score_qas(model, qas)
        save_result(model, run, result)
